In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv(r"F:\Fpoly\smartgen\ai-validation\retail_sales.csv")
print(df.shape)
print(df.head())
print(df.dtypes)

(4565000, 8)
         date store_id item_id  sales  price  promo  weekday  month
0  2019-01-01  store_1  item_1     41  21.30      0        1      1
1  2019-01-02  store_1  item_1     53  21.30      0        2      1
2  2019-01-03  store_1  item_1     39  21.30      0        3      1
3  2019-01-04  store_1  item_1     35  21.30      0        4      1
4  2019-01-05  store_1  item_1     51  17.04      1        5      1
date            str
store_id        str
item_id         str
sales         int64
price       float64
promo         int64
weekday       int64
month         int64
dtype: object


In [2]:
print("Date range:", df["date"].min(), "→", df["date"].max())
print("Số stores:", df["store_id"].nunique())
print("Số items:", df["item_id"].nunique())
print("Số SKU:", df.groupby(["store_id", "item_id"]).ngroups)

Date range: 2019-01-01 → 2023-12-31
Số stores: 50
Số items: 50
Số SKU: 2500


In [3]:
sku = df[(df["store_id"] == "store_1") & (df["item_id"] == "item_1")].copy()
sku["date"] = pd.to_datetime(sku["date"])
sku = sku.sort_values("date").reset_index(drop=True)
print(f"Số ngày data: {len(sku)}")
print(sku[["date", "sales"]].tail())

Số ngày data: 1826
           date  sales
1821 2023-12-27     58
1822 2023-12-28     58
1823 2023-12-29     41
1824 2023-12-30     38
1825 2023-12-31     42


In [4]:
train_size = int(len(sku) * 0.8)

def smape(actual, predicted):
  mask = ~((actual == 0) & (predicted == 0))
  return 100 * np.mean(2 * np.abs(actual[mask] - predicted[mask]) /
					   (np.abs(actual[mask]) + np.abs(predicted[mask])))

In [5]:
from prophet import Prophet

train = sku[["date", "sales"]].rename(columns={"date": "ds", "sales": "y"})
model = Prophet(daily_seasonality=False)
model.fit(train[:train_size])

forecast = model.predict(model.make_future_dataframe(periods=len(sku) - train_size))
score_prophet = smape(train["y"][train_size:].values,
					forecast["yhat"].tail(len(sku) - train_size).values)
print(f"Prophet sMAPE: {score_prophet:.2f}%")

Importing plotly failed. Interactive plots will not work.
13:05:41 - cmdstanpy - INFO - Chain [1] start processing
13:05:41 - cmdstanpy - INFO - Chain [1] done processing


Prophet sMAPE: 9.75%


In [6]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing

train_sales = sku["sales"][:train_size].values
test_sales  = sku["sales"][train_size:].values

try:
  ets_model = ExponentialSmoothing(
	  train_sales, trend='add', seasonal='mul',
	  seasonal_periods=7, damped_trend=True
  ).fit(optimized=True)
except Exception:
  ets_model = ExponentialSmoothing(
	  train_sales, trend='add', seasonal='add',
	  seasonal_periods=7, damped_trend=True
  ).fit(optimized=True)

score_ets = smape(test_sales, ets_model.forecast(len(test_sales)))
print(f"ETS sMAPE: {score_ets:.2f}%")

ETS sMAPE: 23.92%


In [7]:
from xgboost import XGBRegressor

sku_xgb = sku.copy()
for lag in [1, 7, 14]:
  sku_xgb[f"lag_{lag}"] = sku_xgb["sales"].shift(lag)
sku_xgb = sku_xgb.dropna()

features = ["lag_1", "lag_7", "lag_14", "weekday", "month", "price", "promo"]
X = sku_xgb[features].values
y = sku_xgb["sales"].values
n = int(len(X) * 0.8)

xgb_model = XGBRegressor(n_estimators=100, random_state=42)
xgb_model.fit(X[:n], y[:n])
score_xgb = smape(y[n:], xgb_model.predict(X[n:]))
print(f"XGBoost sMAPE: {score_xgb:.2f}%")

XGBoost sMAPE: 8.44%


In [8]:
print("=" * 45)
print("MODEL COMPETITION — store_1 / item_1")
print("=" * 45)
results = {"Prophet": score_prophet, "ETS": score_ets, "XGBoost": score_xgb}
winner = min(results, key=results.get)
for name, score in results.items():
  flag = "<- WINNER" if name == winner else ""
  print(f"{name:10s}: {score:.2f}%  {flag}")
print("=" * 45)

MODEL COMPETITION — store_1 / item_1
Prophet   : 9.75%  
ETS       : 23.92%  
XGBoost   : 8.44%  <- WINNER


In [9]:
test_skus = [
  ("store_1", "item_2"),
  ("store_1", "item_5"),
  ("store_2", "item_1"),
  ("store_3", "item_10"),
]

print(f"{'SKU':<22} {'Prophet':>10} {'ETS':>10} {'XGBoost':>10}  WINNER")
print("-" * 65)

for store, item in test_skus:
    try:
        sku_t = df[(df["store_id"] == store) & (df["item_id"] == item)].copy()
        sku_t["date"] = pd.to_datetime(sku_t["date"])
        sku_t = sku_t.sort_values("date").reset_index(drop=True)
        n = int(len(sku_t) * 0.8)

        # Prophet
        train_p = sku_t[["date","sales"]].rename(columns={"date":"ds","sales":"y"})
        mp = Prophet(daily_seasonality=False)
        mp.fit(train_p[:n])
        s_p = smape(train_p["y"][n:].values,
                    mp.predict(mp.make_future_dataframe(periods=len(sku_t)-n))["yhat"].tail(len(sku_t)-n).values)

        # ETS
        try:
            em = ExponentialSmoothing(sku_t["sales"][:n].values, trend='add',
                                      seasonal='mul', seasonal_periods=7,
                                      damped_trend=True).fit(optimized=True)
        except Exception:
            em = ExponentialSmoothing(sku_t["sales"][:n].values, trend='add',
                                      seasonal='add', seasonal_periods=7).fit(optimized=True)
        s_e = smape(sku_t["sales"][n:].values, em.forecast(len(sku_t)-n))

        # XGBoost — chỉ dùng các cột chắc chắn tồn tại
        sku_x = sku_t.copy()
        sku_x["dayofweek"] = sku_x["date"].dt.dayofweek
        sku_x["month"] = sku_x["date"].dt.month
        for lag in [1, 7, 14]:
            sku_x[f"lag_{lag}"] = sku_x["sales"].shift(lag)
        sku_x = sku_x.dropna()
        feats = ["lag_1", "lag_7", "lag_14", "dayofweek", "month"]
        Xm = sku_x[feats].values
        ym = sku_x["sales"].values
        n2 = int(len(Xm) * 0.8)
        xgb2 = XGBRegressor(n_estimators=100, random_state=42)
        xgb2.fit(Xm[:n2], ym[:n2])
        s_x = smape(ym[n2:], xgb2.predict(Xm[n2:]))

        scores = {"Prophet": s_p, "ETS": s_e, "XGBoost": s_x}
        w = min(scores, key=scores.get)
        sku_label = f"{store}/{item}"
        print(f"{sku_label:<22} {s_p:>9.2f}% {s_e:>9.2f}% {s_x:>9.2f}%  {w}")
    except Exception as e:
        print(f"LỖI tại {store}/{item}: {e}")

print("-" * 65)
print("Hoàn tất.")

SKU                       Prophet        ETS    XGBoost  WINNER
-----------------------------------------------------------------


13:05:56 - cmdstanpy - INFO - Chain [1] start processing
13:05:56 - cmdstanpy - INFO - Chain [1] done processing


store_1/item_2             10.92%     36.30%     15.04%  Prophet


13:05:57 - cmdstanpy - INFO - Chain [1] start processing
13:05:57 - cmdstanpy - INFO - Chain [1] done processing


store_1/item_5             11.44%     24.62%     14.22%  Prophet


13:05:58 - cmdstanpy - INFO - Chain [1] start processing
13:05:58 - cmdstanpy - INFO - Chain [1] done processing


store_2/item_1              8.90%     29.57%     11.83%  Prophet


13:05:59 - cmdstanpy - INFO - Chain [1] start processing
13:05:59 - cmdstanpy - INFO - Chain [1] done processing


store_3/item_10            12.00%     23.62%     17.04%  Prophet
-----------------------------------------------------------------
Hoàn tất.
